In [5]:
# Import necessary libraries
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from warnings import filterwarnings
filterwarnings('ignore')

pd.set_option('display.max_columns', None)

In [6]:
# Load the dataset
raw_data = pd.read_csv('Data/postings.csv')
df = raw_data.copy()
df.head()

,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,min_salary,formatted_work_type,applies,original_listed_time,remote_allowed,job_posting_url,application_url,application_type,expiry,closed_time,formatted_experience_level,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,17.0,Full-time,2.0,1.710000e+12,NaN,https://www.linkedin.com/jobs/view/921716/?trk...,NaN,ComplexOnsiteApply,1.720000e+12,NaN,NaN,Requirements: \n\nWe are seeking a College or ...,1.710000e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,38480.0,8540.0,34021.0
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,30.0,Full-time,NaN,1.710000e+12,NaN,https://www.linkedin.com/jobs/view/1829192/?tr...,NaN,ComplexOnsiteApply,1.720000e+12,NaN,NaN,NaN,1.710000e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,83200.0,80521.0,8069.0
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,45000.0,Full-time,NaN,1.710000e+12,NaN,https://www.linkedin.com/jobs/view/10998357/?t...,NaN,ComplexOnsiteApply,1.720000e+12,NaN,NaN,We are currently accepting resumes for FOH - A...,1.710000e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,55000.0,45202.0,39061.0
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,175000.0,YEARLY,"New Hyde Park, NY",766262.0,16.0,NaN,140000.0,Full-time,NaN,1.710000e+12,NaN,https://www.linkedin.com/jobs/view/23221523/?t...,NaN,ComplexOnsiteApply,1.720000e+12,NaN,NaN,This position requires a baseline understandin...,1.710000e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,157500.0,11040.0,36059.0
4,35982263,NaN,Service Technician,Looking for HVAC service tech with experience ...,80000.0,YEARLY,"Burlington, IA",NaN,3.0,NaN,60000.0,Full-time,NaN,1.710000e+12,NaN,https://www.linkedin.com/jobs/view/35982263/?t...,NaN,ComplexOnsiteApply,1.720000e+12,NaN,NaN,NaN,1.710000e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,70000.0,52601.0,19057.0


In [7]:
# Explore the dataset
print(f"Rows: {df.shape[0]}")
print("-"*20)
print(f"\nColumns: {df.shape[1]}")
print("-"*20)
print(f"\nColumns: {df.columns.tolist()}")

Rows: 123849
--------------------

Columns: 31
--------------------

Columns: ['job_id', 'company_name', 'title', 'description', 'max_salary', 'pay_period', 'location', 'company_id', 'views', 'med_salary', 'min_salary', 'formatted_work_type', 'applies', 'original_listed_time', 'remote_allowed', 'job_posting_url', 'application_url', 'application_type', 'expiry', 'closed_time', 'formatted_experience_level', 'skills_desc', 'listed_time', 'posting_domain', 'sponsored', 'work_type', 'currency', 'compensation_type', 'normalized_salary', 'zip_code', 'fips']


In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 31 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   job_id                      123849 non-null  int64  
 1   company_name                122130 non-null  str    
 2   title                       123849 non-null  str    
 3   description                 123842 non-null  str    
 4   max_salary                  29793 non-null   float64
 5   pay_period                  36073 non-null   str    
 6   location                    123849 non-null  str    
 7   company_id                  122132 non-null  float64
 8   views                       122160 non-null  float64
 9   med_salary                  6280 non-null    float64
 10  min_salary                  29793 non-null   float64
 11  formatted_work_type         123849 non-null  str    
 12  applies                     23320 non-null   float64
 13  original_listed_time     

In [14]:
# Fix data types
count_cols = ['company_id', 'views', 'applies', 'remote_allowed']
for col in count_cols:
    df[col] = df[col].astype('Int64')

time_cols = ['original_listed_time', 'expiry', 'closed_time', 'listed_time']
for col in time_cols:
    df[col] = pd.to_datetime(df[col], unit='ms', errors='coerce')

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 31 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   job_id                      123849 non-null  int64         
 1   company_name                122130 non-null  str           
 2   title                       123849 non-null  str           
 3   description                 123842 non-null  str           
 4   max_salary                  29793 non-null   float64       
 5   pay_period                  36073 non-null   str           
 6   location                    123849 non-null  str           
 7   company_id                  122132 non-null  Int64         
 8   views                       122160 non-null  Int64         
 9   med_salary                  6280 non-null    float64       
 10  min_salary                  29793 non-null   float64       
 11  formatted_work_type         123849 non-null  str  

In [15]:
df.sample(10)

,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,min_salary,formatted_work_type,applies,original_listed_time,remote_allowed,job_posting_url,application_url,application_type,expiry,closed_time,formatted_experience_level,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
86303,3904381846,"PLS Financial Services, Inc.",Customer Service Representative- Overnight Shift,Overview\n\n PLS ® : People. Location. Service...,NaN,NaN,"Pasadena, TX",23929,5,NaN,NaN,Full-time,<NA>,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3904381846/...,https://careers-pls.icims.com/jobs/21901/custo...,OffsiteApply,2024-07-03 09:46:40,NaT,Mid-Senior level,NaN,2024-03-09 16:00:00,careers-pls.icims.com,0,FULL_TIME,NaN,NaN,NaN,77502.0,48201.0
96368,3904946342,Cape Fear Valley Health,Nursing Supervisor,Facility\n\nBetsy Johnson Hospital\n\nLocation...,NaN,NaN,"Dunn, NC",42179,4,NaN,NaN,Full-time,<NA>,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3904946342/...,https://capefearvalley.wd1.myworkdayjobs.com/C...,OffsiteApply,2024-07-03 09:46:40,NaT,Mid-Senior level,NaN,2024-03-09 16:00:00,capefearvalley.wd1.myworkdayjobs.com,0,FULL_TIME,NaN,NaN,NaN,28334.0,37085.0
48246,3901344534,XL Pro Staffing and Consulting Group,Physical Security Manager,Vision Statement:At XL Pro Staffing & Consulti...,156000.00,YEARLY,"Wilmer, TX",9391120,9,NaN,95000.00,Full-time,<NA>,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3901344534/...,NaN,ComplexOnsiteApply,2024-07-03 09:46:40,NaT,Mid-Senior level,NaN,2024-03-09 16:00:00,NaN,0,FULL_TIME,USD,BASE_SALARY,125500.00,75172.0,48113.0
95810,3904937425,BioTouch,Temporary Assistant,NOT a real post - testing only - do not apply,NaN,NaN,"Atlanta, GA",89685467,7,NaN,NaN,Temporary,<NA>,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3904937425/...,NaN,ComplexOnsiteApply,2024-07-03 09:46:40,2024-03-09 16:00:00,Internship,NaN,2024-03-09 16:00:00,NaN,0,TEMPORARY,NaN,NaN,NaN,30303.0,13121.0
87989,3904392059,CSM Corporation,Housekeeping Associate,This position is responsible for cleaning gues...,NaN,NaN,"Minneapolis, MN",428055,6,NaN,NaN,Full-time,<NA>,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3904392059/...,https://recruiting2.ultipro.com/CSM1000CSMM/Jo...,OffsiteApply,2024-07-03 09:46:40,NaT,Entry level,NaN,2024-03-09 16:00:00,recruiting2.ultipro.com,0,FULL_TIME,NaN,NaN,NaN,55401.0,27053.0
31475,3894679338,Barcadia Bar & Grill,Line Cook,"Location: New Orleans, LA\n\nEmployment Type: ...",NaN,NaN,"New Orleans, LA",54814820,2,NaN,NaN,Full-time,<NA>,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3894679338/...,https://barcadianola.companycareersite.com/Job...,OffsiteApply,2024-07-03 09:46:40,NaT,Entry level,NaN,2024-03-09 16:00:00,barcadianola.companycareersite.com,0,FULL_TIME,NaN,NaN,NaN,70112.0,22071.0
67000,3902751645,Sharp Decisions,Attorney (W2 and Phoenix candidates),POSITION DESCRIPTION:The Office of the General...,33.87,YEARLY,"Phoenix, AZ",26376,6,NaN,33.87,Contract,<NA>,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3902751645/...,NaN,ComplexOnsiteApply,2024-07-03 09:46:40,NaT,Associate,NaN,2024-03-09 16:00:00,NaN,0,CONTRACT,USD,BASE_SALARY,33.87,85003.0,4013.0
25375,3891074281,Blum USA,Business Development Manager,"About BlumThe Blum name stands for quality, in...",NaN,NaN,"Stanley, NC",10367916,2,NaN,NaN,Full-time,<NA>,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3891074281/...,https://workforcenow.adp.com/mascsr/default/md...,OffsiteApply,2024-07-03 09:46:40,NaT,NaN,NaN,2024-03-09 16:00:00,NaN,0,FULL_TIME,NaN,NaN,NaN,28164.0,37071.0
114655,3905832626,Net2Source Inc.,Senior Auditor,Skills needed:Looking for some insurance backg...,51.00,HOURLY,"Boston, MA",226965,4,NaN,50.00,Contract,<NA>,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3905832626/...,NaN,ComplexOnsiteApply,2024-07-03 09:46:40,NaT,M

In [17]:
# Standardize column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df.sample(10)

,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,min_salary,formatted_work_type,applies,original_listed_time,remote_allowed,job_posting_url,application_url,application_type,expiry,closed_time,formatted_experience_level,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
18689,3888943421,Weatherby Healthcare,Locum | Physician Anesthesiology,Enjoy the locum tenens lifestyle knowing Weath...,NaN,NaN,"Aurora, IL",45530,3,NaN,NaN,Full-time,<NA>,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3888943421/...,https://jsv3.recruitics.com/redirect?rx_cid=43...,OffsiteApply,2024-07-03 09:46:40,NaT,Mid-Senior level,NaN,2024-03-09 16:00:00,jsv3.recruitics.com,0,FULL_TIME,NaN,NaN,NaN,60502.0,NaN
113793,3905696613,Gryphon Oakwood,"Vice President, Land Acquisition - Luxury Res...","Vice President, Land Acquisition - Luxury Resi...",NaN,NaN,"Florida, United States",89290788,5,NaN,NaN,Full-time,<NA>,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3905696613/...,NaN,ComplexOnsiteApply,2024-07-03 09:46:40,NaT,Executive,NaN,2024-03-09 16:00:00,NaN,0,FULL_TIME,NaN,NaN,NaN,NaN,NaN
89289,3904409867,Randstad Digital Americas,Enterprise Application Engineer,Job Summary\n\nAre you ready to elevate your c...,NaN,NaN,"Hackensack, NJ",157324,15,NaN,NaN,Full-time,<NA>,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3904409867/...,https://www.randstadusa.com/jobs/4/1049257/ent...,OffsiteApply,2024-07-03 09:46:40,NaT,NaN,NaN,2024-03-09 16:00:00,www.randstadusa.com,0,FULL_TIME,NaN,NaN,NaN,7601.0,34003.0
120130,3906228027,RWJBarnabas Health,"Registered Nurse (RN), Operating Room",Req #: 0000149021\n\nCategory: OR Nursing\n\...,NaN,NaN,"New Brunswick, NJ",14363,4,NaN,NaN,Part-time,<NA>,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3906228027/...,https://www.rwjbarnabashealthcareers.org/job/r...,OffsiteApply,2024-07-03 09:46:40,NaT,Mid-Senior level,NaN,2024-03-09 16:00:00,www.rwjbarnabashealthcareers.org,0,PART_TIME,NaN,NaN,NaN,8901.0,34023.0
9282,3886477759,GREENBRIDGE,ASSISTANT PROPERTY MANAGER,Assistant Property ManagerThe Assistant Proper...,75000.0,YEARLY,"Seattle, WA",2622009,37,NaN,65000.0,Full-time,5,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3886477759/...,NaN,ComplexOnsiteApply,2024-10-27 03:33:20,NaT,NaN,NaN,2024-03-09 16:00:00,NaN,0,FULL_TIME,USD,BASE_SALARY,70000.0,98101.0,53033.0
107202,3905331689,"BAE Systems, Inc.",Dynamics and Controls Engineer,\nJob Description What does it take to push th...,156860.0,YEARLY,"Sterling Heights, MI",1881,4,NaN,92290.0,Full-time,<NA>,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3905331689/...,https://jobs.baesystems.com/global/en/job/BAE1...,OffsiteApply,2024-07-03 09:46:40,NaT,Entry level,NaN,2024-03-09 16:00:00,jobs.baesystems.com,0,FULL_TIME,USD,BASE_SALARY,124575.0,48310.0,26099.0
88169,3904392751,"JELD-WEN, Inc.",Production Mgr,JELD-WEN is currently seeking a Production Mgr...,NaN,NaN,"Sterling Heights, MI",15201,6,NaN,NaN,Full-time,<NA>,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3904392751/...,https://jobs.jeld-wen.com/job/Sterling-Heights...,OffsiteApply,2024-07-03 09:46:40,NaT,Mid-Senior level,NaN,2024-03-09 16:00:00,jobs.jeld-wen.com,0,FULL_TIME,NaN,NaN,NaN,48310.0,26099.0
7119,3885825307,Elara Caring,Occupational Therapist OT Home Health PRN,"At Elara Caring, we have a unique opportunity ...",NaN,NaN,"Jefferson City, MO",11818084,4,NaN,NaN,Part-time,<NA>,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3885825307/...,https://careers.elara.com/us/en/job/ECYECHUSJR...,OffsiteApply,2024-03-09 16:00:00,NaT,Entry level,NaN,2024-03-09 16:00:00,careers.elara.com,0,PART_TIME,NaN,NaN,NaN,65101.0,29051.0
46574,3900975642,Markham Vineyards,Busser,We have an exciting opportunity for temporary ...,19.5,HOURLY,"St Helena, CA",5025769,21,NaN,18.0,Temporary,1,2024-03-09 16:00:00,<NA>,https://www.link

In [18]:
# check for duplicates
df.duplicated().sum()

np.int64(0)

In [ ]:
df[df.duplicated(subset=['job_id'], keep=False)]

,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,min_salary,formatted_work_type,applies,original_listed_time,remote_allowed,job_posting_url,application_url,application_type,expiry,closed_time,formatted_experience_level,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips


In [21]:
# Check for missing values
df.isnull().sum()

job_id                             0
company_name                    1719
title                              0
description                        7
max_salary                     94056
pay_period                     87776
location                           0
company_id                      1717
views                           1689
med_salary                    117569
min_salary                     94056
formatted_work_type                0
applies                       100529
original_listed_time               0
remote_allowed                108603
job_posting_url                    0
application_url                36665
application_type                   0
expiry                             0
closed_time                   122776
formatted_experience_level     29409
skills_desc                   121410
listed_time                        0
posting_domain                 39968
sponsored                          0
work_type                          0
currency                       87776
c